# Forward hooks (how look.py sees inside)

**Previous:** [07_logits](07_logits.ipynb)  
**Home:** [../00_START_HERE.ipynb](../00_START_HERE.ipynb)  
**Next:** [09_evolution_through_transformer](09_evolution_through_transformer.ipynb) · then [02_observe overview](../02_observe/00_OVERVIEW.ipynb)

**Kernel:** `CXR local Qwen (faiss_gpu1)`  
**Status:** executable (calls `scripts/` — same as CLI)

---

## 1. Why am I learning this?

HF `register_forward_hook` is the camera. No TransformerLens required for CXR science.

## 2. Mental model

Hook fires during forward; we copy last-token activations to CPU.

## 3. Clinical question

Can I capture last-token at the CXR-prior layer (default L20) with a 5-line hook?

## 4. Prediction

Yes — shape (3584,).


## 5. Minimal Python — setup + load (run once)


In [1]:
# Shared bootstrap — run this first in every executable notebook
import sys
from pathlib import Path

# notebooks/_lib regardless of how deep this notebook sits
_here = Path.cwd().resolve()
for _p in [_here, *_here.parents]:
    if (_p / "_lib" / "cxr_boot.py").is_file():
        sys.path.insert(0, str(_p / "_lib"))
        break
    if (_p / "notebooks" / "_lib" / "cxr_boot.py").is_file():
        sys.path.insert(0, str(_p / "notebooks" / "_lib"))
        break
else:
    raise RuntimeError("Cannot find notebooks/_lib/cxr_boot.py — open Jupyter with notebooks/ as root")

import cxr_boot
ctx = cxr_boot.setup(layer=20, max_new=24, load_model=True)
model, tok = ctx["model"], ctx["tok"]
NOTE, LAYER, MAX_NEW = ctx["NOTE"], ctx["LAYER"], ctx["MAX_NEW"]
PROMPT_A, PROMPT_B, PROMPT_TEST = ctx["PROMPT_A"], ctx["PROMPT_B"], ctx["PROMPT_TEST"]
look, intervene, process = ctx["look"], ctx["intervene"], ctx["process"]
print("NOTE:", NOTE)
print("LAYER:", LAYER, "ready")


load Qwen/Qwen2.5-7B-Instruct (local only) …


loading file vocab.json from cache at /home/udonsi-kalu/.cache/huggingface/hub/models--Qwen--Qwen2.5-7B-Instruct/snapshots/a09a35458c702b33eeacc393d103063234e8bc28/vocab.json
loading file merges.txt from cache at /home/udonsi-kalu/.cache/huggingface/hub/models--Qwen--Qwen2.5-7B-Instruct/snapshots/a09a35458c702b33eeacc393d103063234e8bc28/merges.txt
loading file tokenizer.json from cache at /home/udonsi-kalu/.cache/huggingface/hub/models--Qwen--Qwen2.5-7B-Instruct/snapshots/a09a35458c702b33eeacc393d103063234e8bc28/tokenizer.json
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at None
loading file tokenizer_config.json from cache at /home/udonsi-kalu/.cache/huggingface/hub/models--Qwen--Qwen2.5-7B-Instruct/snapshots/a09a35458c702b33eeacc393d103063234e8bc28/tokenizer_config.json
loading file chat_template.jinja from cache at None
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or tra

Loading checkpoint shards: 100%|██████████| 4/4 [00:01<00:00,  2.37it/s]

All model checkpoint weights were used when initializing Qwen2ForCausalLM.

All the weights of Qwen2ForCausalLM were initialized from the model checkpoint at Qwen/Qwen2.5-7B-Instruct.
If your task is similar to the task the model of the checkpoint was trained on, you can already use Qwen2ForCausalLM for predictions without further training.
loading configuration file generation_config.json from cache at /home/udonsi-kalu/.cache/huggingface/hub/models--Qwen--Qwen2.5-7B-Instruct/snapshots/a09a35458c702b33eeacc393d103063234e8bc28/generation_config.json
Generate config GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "repetition_penalty": 1.05,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.8
}




BACKEND  FRESH LOAD complete — inspecting live weights
  pid=56120  class=Qwen2ForCausalLM
  name=Qwen/Qwen2.5-7B-Instruct
  torch_dtype=torch.float16  training=False
  blocks=28  hidden=3584  using layer 20
  tokenizer=Qwen2TokenizerFast  vocab=151643  pad=151643  eos=151645
  hf_device_map (accelerate placement):
    : cpu
  device_map counts: {'cpu': 1}
  param tensors by device: {'cpu': 339}
  param dtypes: {'torch.float16': 339}
  params=7.616B  weight_GiB=14.19
  cuda_alloc_GiB=0.00  cuda_reserved_GiB=0.00
  cuda0 free_GiB=0.63  total_GiB=23.56
NOTE: Patient received FOLFOX. Disease progressed. FOLFOX was discontinued.
LAYER: 20 ready


## 6. Run


In [2]:
from _common import blocks, unwrap, encode, device
import torch

ids = encode(tok, NOTE, device(model))
captured = {}
layer = blocks(model)[LAYER]

def hook(_m, _inp, out):
    captured["h"] = unwrap(out)[0, -1, :].detach().float().cpu()

hdl = layer.register_forward_hook(hook)
try:
    with torch.inference_mode():
        model(**ids)
finally:
    hdl.remove()

h = captured["h"]
print("hooked L20 last-token", tuple(h.shape), "L2", float(h.norm()))


hooked L20 last-token (3584,) L2 114.62896728515625


## 7–8. What happened / What did I learn?

_(fill after you run)_

## 9. Claim boundary

✓ We attached and removed a forward hook successfully.

✗ Hooking ≠ intervening; we did not modify the forward.

## 10. CXR connection

`scripts/_common.py` · foundations of all look modes

## 11. Questions

## 12. Revision notes

| Date | Change |
|------|--------|
| | |
